<a href="https://colab.research.google.com/github/lPragathi/BANKING-BOT/blob/main/Banking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install openai langchain-community langchain-huggingface langchain-text-splitters faiss-cpu sentence-transformers pypdf

import requests
from openai import OpenAI
from google.colab import userdata
from langchain_core.documents import Document
from langchain_community.document_loaders import WebBaseLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# ---------------- DATA ----------------

data = [
    "UPI is an instant payment system that allows users to transfer money between bank accounts using UPI IDs, mobile numbers and QR codes.",
    "IMPS is an instant interbank electronic fund transfer service operated by NPCI and is available 24 hours a day.",
    "NEFT is an electronic funds transfer system operated by the Reserve Bank of India.",
    "RTGS is a real-time electronic fund transfer system mainly used for high-value transactions.",
    "Customers should never share their OTP, UPI PIN, ATM PIN, CVV or banking password with anyone.",
    "For an unauthorised transaction, customers should immediately contact their bank through official channels.",
    "Customers should verify the beneficiary, account number, IFSC, UPI ID and transaction amount before making a payment."
]

docs = [Document(page_content=x) for x in data]

# ---------------- RBI WEBSITE ----------------

try:
    web = WebBaseLoader(
        web_paths=["https://www.rbi.org.in/"]
    ).load()
    docs += web
except:
    pass

# ---------------- RBI BROCHURE ----------------

try:
    url = "https://systemhealth.rbi.org.in/cms.rbi.org.in/cms/assets/Documents/BEAWARE07032022.pdf"
    pdf = "/content/rbi.pdf"

    r = requests.get(url, timeout=20)
    open(pdf, "wb").write(r.content)

    docs += PyPDFLoader(pdf).load()
except:
    pass

# ---------------- VECTOR DATABASE ----------------

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(docs)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

db = FAISS.from_documents(chunks, embeddings)

# ---------------- OPENROUTER ----------------

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=userdata.get("OPENROUTER_API_KEY")
)

# ---------------- ASK ONE QUESTION ----------------

question = input("Enter your banking question: ")

results = db.similarity_search(question, k=3)

context = "\n\n".join(
    x.page_content for x in results
)

prompt = f"""
You are a banking information assistant.

Answer ONLY using the context below.
Do not guess.
If the answer is not available, say:
"I do not have enough information."

Never ask for OTP, PIN, CVV or password.

CONTEXT:
{context}
"""

response = client.chat.completions.create(
    model="meta-llama/llama-3.3-70b-instruct",
    messages=[
        {"role": "system", "content": prompt},
        {"role": "user", "content": question}
    ],
    temperature=0
)

print("\n🤖 AI:", response.choices[0].message.content)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.7/401.7 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


/tmp/ipykernel_7753/1921294931.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader, PyPDFLoader


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Enter your banking question: TELL ABOUT TRANSACTIONS

🤖 AI: To ensure secure transactions, customers should verify the following details before making a payment:

1. Beneficiary
2. Account number
3. IFSC (if applicable)
4. UPI ID (if applicable)
5. Transaction amount

This verification process helps prevent errors and unauthorized transactions.
